In [2]:
import os
import json
import ollama

# 1. ตั้งค่า Path ของไฟล์ Text ที่ได้จาก EasyOCR
txt_folder = '/Users/faiijaran/Desktop/project/output/extracted_text'

def ask_llama(text_content):
    """ส่งข้อความที่เละจาก OCR ให้ Llama 3.2 ในเครื่องช่วยซ่อมและสรุปผล"""
    prompt = f"""
    คุณคือผู้เชี่ยวชาญด้านบัญชี (Expert Accountant) 
    ด้านล่างนี้คือข้อความที่ได้จากใบเสร็จผ่านระบบ OCR ซึ่งมีคำสะกดผิดและ Noise เยอะมาก
    
    หน้าที่ของคุณ:
    1. วิเคราะห์และซ่อมแซมคำที่ผิดตามบริบท (เช่น '6kk mu' คือ 'MR.D.I.Y.')
    2. สกัดข้อมูลเป็น JSON เท่านั้น โดยมี Key ดังนี้:
       - company: ชื่อบริษัทหรือร้านค้า
       - date: วันที่ในใบเสร็จ (รูปแบบ YYYY-MM-DD)
       - total: ยอดรวมสุทธิ (เป็นตัวเลขทศนิยม Float)
    
    ข้อความ OCR:
    {text_content}
    """
    
    # เรียกใช้ Llama 3.2 ที่ติดตั้งสำเร็จแล้วในเครื่องคุณ
    response = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': prompt}],
        format='json'
    )
    return response['message']['content']

# 2. เริ่มวนลูปประมวลผลทั้ง 16 ไฟล์
print("🤖 Llama 3.2 กำลังเริ่มวิเคราะห์ข้อมูล (Local Machine)...")
print("=" * 50)

for i in range(1, 17):
    filename = f"text_{i}.txt"
    file_path = os.path.join(txt_folder, filename)
    
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                raw_text = f.read()
            
            # ส่งให้ AI วิเคราะห์ (ขั้นตอนนี้อาจใช้เวลาสักครู่ต่อไฟล์)
            json_result = ask_llama(raw_text)
            data = json.loads(json_result)
            
            # --- แสดงผลลัพธ์ออกที่หน้าจอ ---
            print(f"📍 [ผลลัพธ์จากไฟล์: {filename}]")
            print(json.dumps(data, ensure_ascii=False, indent=4))
            print("-" * 30)
            
        except Exception as e:
            print(f"❌ เกิดข้อผิดพลาดที่ไฟล์ {filename}: {e}")
    else:
        print(f"⚠️ ไม่พบไฟล์: {filename}")

print("\n✨ ตรวจสอบผลลัพธ์ด้านบนได้เลยครับ!")

🤖 Llama 3.2 กำลังเริ่มวิเคราะห์ข้อมูล (Local Machine)...
📍 [ผลลัพธ์จากไฟล์: text_1.txt]
{
    "company": "MR.D.I.Y.",
    "date": "2020-01-11",
    "total": 1.33
}
------------------------------
📍 [ผลลัพธ์จากไฟล์: text_2.txt]
{
    "company": "MR.D.I.Y.",
    "date": "2022-07-14",
    "total": 103.27
}
------------------------------
📍 [ผลลัพธ์จากไฟล์: text_3.txt]
{
    "company": "โลทัส ซัมมาร์ท์ เอปพลิอาม",
    "date": "2022-01-02",
    "total": 6.0
}
------------------------------
📍 [ผลลัพธ์จากไฟล์: text_4.txt]
{
    "company": "บริษัทหรือร้านค้า",
    "date": "2565-05-15",
    "total": 999.99
}
------------------------------
📍 [ผลลัพธ์จากไฟล์: text_5.txt]
{
    "company": "MR.D.I.Y.",
    "date": "12.5/2563",
    "total": 2250.5
}
------------------------------
📍 [ผลลัพธ์จากไฟล์: text_6.txt]
{
    "company": "axtra pcl หาดาหญ่",
    "date": "02-01-2026",
    "total": 153.0
}
------------------------------
📍 [ผลลัพธ์จากไฟล์: text_7.txt]
{
    "company": "",
    "date": "",
    "total

In [3]:
import os
import json
import ollama

# 1. ตั้งค่า Path ของไฟล์ Text ที่ได้จาก EasyOCR
txt_folder = '/Users/faiijaran/Desktop/project/output/extracted_text'

def ask_llama(text_content):
    prompt = f"""
    คุณคือผู้เชี่ยวชาญด้านบัญชีไทย หน้าที่ของคุณคือสกัดข้อมูลจาก OCR ที่ "เละ" ให้เป็น JSON ที่ถูกต้อง
    
    กฎเหล็ก:
    1. หากเป็นปี พ.ศ. (25xx) ให้ลบ 543 เพื่อเปลี่ยนเป็น ค.ศ. (20xx) เสมอ
    2. ยอดรวม (total) ต้องเป็นตัวเลขที่มีทศนิยมไม่เกิน 2 ตำแหน่งเท่านั้น
    3. ชื่อบริษัทต้องเป็นชื่อเฉพาะ (เช่น Lotus, Big C, MR.D.I.Y) หากหาไม่เจอจริงๆ ให้ใส่ "Unknown"
    
    ตัวอย่างการซ่อมคำ:
    - '6kk mu' หรือ 'mi. d' -> 'MR.D.I.Y.'
    - 'axtra' หรือ 'cp axtra' -> 'CP Axtra'
    - '2569-01-01' -> '2026-01-01'
    
    ข้อความ OCR:
    {text_content}
    """
    
    response = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': prompt}],
        format='json'
    )
    return response['message']['content']

# 2. เริ่มวนลูปประมวลผลทั้ง 16 ไฟล์
print("🤖 Llama 3.2 กำลังเริ่มวิเคราะห์ข้อมูล (Local Machine)...")
print("=" * 50)

for i in range(1, 17):
    filename = f"text_{i}.txt"
    file_path = os.path.join(txt_folder, filename)
    
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                raw_text = f.read()
            
            # ส่งให้ AI วิเคราะห์ (ขั้นตอนนี้อาจใช้เวลาสักครู่ต่อไฟล์)
            json_result = ask_llama(raw_text)
            data = json.loads(json_result)
            
            # --- แสดงผลลัพธ์ออกที่หน้าจอ ---
            print(f"📍 [ผลลัพธ์จากไฟล์: {filename}]")
            print(json.dumps(data, ensure_ascii=False, indent=4))
            print("-" * 30)
            
        except Exception as e:
            print(f"❌ เกิดข้อผิดพลาดที่ไฟล์ {filename}: {e}")
    else:
        print(f"⚠️ ไม่พบไฟล์: {filename}")

print("\n✨ ตรวจสอบผลลัพธ์ด้านบนได้เลยครับ!")

🤖 Llama 3.2 กำลังเริ่มวิเคราะห์ข้อมูล (Local Machine)...
📍 [ผลลัพธ์จากไฟล์: text_1.txt]
{
    "ข้อความ OCR": "(6kk) mu. .เi. i.i./ hr reinl festtil aitat mileh diะ 1519/1-2, 151i 4.303-304, sw flii ไอมภิิามทลา hat iai, hit. vii ni. ส้หบขเม่ s01a0 ก2-i3s-701 โลล 0105551132529 tik ii lia 512901000241401 pis ii necetttu wmniee (an) prico afount escrintion ity hiuk #|-0127. 2u/เร กุากรียเเก ad wi... l x ity(s) iton(s) thi mit tata) ing) thi มิ วทีมท ทรรุ tl aoods 19:11 3|02 4$09 11-01-20 peuti ffhve- tapmmp exphaae aif allied แโถมโอ ไอ้ iโตา neceipt stuttt in ตรห เรฟฟูி uat ine)udnd nhi. irdiv.o. th holsite facehuok ai iii instajra: arliy. thailand .uraiythailani lin loutube: it iti thailand"
}
------------------------------
📍 [ผลลัพธ์จากไฟล์: text_2.txt]
{
    "ชื่อบริษัท": "Tatal Inci.",
    "ที่อยู่": "ไม่พบ",
    "เบอร์โทรศัพท์": "42-136-2401",
    "วันที่": "2026-01-01",
    "จำนวนเงิน": "174519.00 บาท",
    "ราคารวม": "183,000.00 บาท",
    "ภาษีมูลค่าเพิ่ม": "13,200.00 บาท"
}
-------